# Etapa 4 — De dónde salen los pesos

**Introducción a la Ciencia de Datos · Equipo Videojuegos**

En la etapa 3 mostramos una tabla con los pesos de cada atributo sobre el OVR, posición
por posición. La devolución fue clara: **los números estaban, pero no se veía de dónde
salían.**

Este notebook responde esa pregunta y nada más. No agrega modelos nuevos: desarma el que
ya teníamos hasta llegar al caso más simple posible —una sola variable, una recta en un
gráfico— y vuelve a subir desde ahí hasta la tabla de la etapa 3.

El recorrido es éste:

1. **Qué es un peso.** Con un solo atributo, el peso es la pendiente de una recta.
2. **Qué quiere decir "la recta que mejor pasa".** Los errores, dibujados uno por uno.
3. **De dónde sale el número.** Nadie lo elige: es el mínimo de una curva.
4. **Dos atributos.** Deja de ser una recta y pasa a ser un plano, con dos pendientes.
5. **De dos a veintinueve.** Por qué el peso de un atributo baja cuando se agregan otros.
6. **Por qué un modelo por posición.** El mismo atributo con pendientes muy distintas.

Al final de ese recorrido, cada celda de la tabla de la etapa 3 es un número que se puede
señalar en un gráfico.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

DIR_BASE = Path.cwd()
for carpeta in [DIR_BASE, *DIR_BASE.parents]:
    if (carpeta / "data" / "processed").exists():
        DIR_BASE = carpeta
        break

DIR_LIMPIO = DIR_BASE / "data" / "processed"
DIR_FIGURAS = DIR_BASE / "output" / "figuras"
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

RUTA = DIR_LIMPIO / "jugadores_campo_limpio.csv"
if not RUTA.exists():
    raise FileNotFoundError(
        f"No encuentro {RUTA}.\n"
        "Corré primero el notebook 01_carga_limpieza_orden.ipynb, que genera los CSV limpios."
    )

campo = pd.read_csv(RUTA, encoding="utf-8")
print(f"Jugadores de campo: {len(campo):,}".replace(",", "."))

In [ ]:
SUPERFICIE, TINTA = "#fcfcfb", "#0b0b0b"
TINTA_SUAVE, TINTA_TENUE, GRILLA = "#52514e", "#898781", "#e1e0d9"
AZUL, AZUL_CLARO, NARANJA, AQUA, GRIS = "#2a78d6", "#b7d3f6", "#eb6834", "#1baf7a", "#c3c2b7"

mpl.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE,
    "savefig.facecolor": SUPERFICIE, "font.family": "sans-serif", "font.size": 11,
    "text.color": TINTA, "axes.labelcolor": TINTA_SUAVE, "axes.edgecolor": GRILLA,
    "xtick.color": TINTA_TENUE, "ytick.color": TINTA_TENUE,
    "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRILLA, "grid.linewidth": 0.8,
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

ATRIBUTOS = [
    "acceleration", "sprint_speed", "positioning", "finishing", "shot_power",
    "long_shots", "volleys", "penalties", "vision", "crossing", "free_kick_accuracy",
    "short_passing", "long_passing", "curve", "dribbling", "agility", "balance",
    "reactions", "ball_control", "composure", "interceptions", "heading_accuracy",
    "def_awareness", "standing_tackle", "sliding_tackle", "jumping", "stamina",
    "strength", "aggression",
]

NOMBRES = {
    "reactions": "Reacciones", "short_passing": "Pase corto", "finishing": "Definición",
    "ball_control": "Control del balón", "positioning": "Posicionamiento",
    "standing_tackle": "Entrada", "composure": "Serenidad",
}


def poner_titulo(ax, titulo, subtitulo, alto=(1.14, 1.045)):
    ax.text(0, alto[0], titulo, transform=ax.transAxes, fontsize=14,
            fontweight="bold", color=TINTA, va="bottom", ha="left")
    ax.text(0, alto[1], subtitulo, transform=ax.transAxes, fontsize=10.5,
            color=TINTA_SUAVE, va="bottom", ha="left")


def coma(valor, decimales=2):
    """0.723 -> '0,72' (separador decimal rioplatense)."""
    return f"{valor:.{decimales}f}".replace(".", ",")


def miles(entero):
    return f"{int(entero):,}".replace(",", ".")


def guardar(fig, nombre):
    fig.savefig(DIR_FIGURAS / nombre)
    print(f"✓ {nombre}")

## Parte 1 — Qué es un peso

Empecemos por el caso más chico que existe: **un solo atributo.**

Si tuviéramos que adivinar el OVR de un jugador mirando nada más que sus reacciones,
¿qué haríamos? Poner los 15.859 jugadores en un gráfico —reacciones en el eje
horizontal, OVR en el vertical— y trazar la recta que mejor pase por esa nube.

Esa recta tiene una fórmula:

```latex
OVR = intercepto + peso \times reacciones
```

**El peso es la pendiente de esa recta**: cuánto sube el OVR por cada punto que sube el
atributo. No es nada más que eso.

In [ ]:
x = campo[["reactions"]]
y = campo["ovr"]

modelo_1 = LinearRegression().fit(x, y)
peso_1 = modelo_1.coef_[0]
intercepto_1 = modelo_1.intercept_
r2_1 = modelo_1.score(x, y)

print(f"OVR = {intercepto_1:.3f} + {peso_1:.4f} × reacciones")
print(f"R² = {r2_1:.4f}")
print(f"\nUn jugador con 70 de reacciones: {intercepto_1 + peso_1 * 70:.2f} de OVR")
print(f"Un jugador con 71 de reacciones: {intercepto_1 + peso_1 * 71:.2f} de OVR")
print(f"Diferencia: {peso_1:.4f}  <- el peso es exactamente esto")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.8))

muestra = campo.sample(4000, random_state=42)
ax.scatter(muestra["reactions"], muestra["ovr"], s=6, alpha=0.13,
           color=AZUL, edgecolors="none", zorder=2)

rango = np.array([campo["reactions"].min(), campo["reactions"].max()])
ax.plot(rango, intercepto_1 + peso_1 * rango, color=NARANJA, linewidth=2.6, zorder=4)

# El escalon que muestra que la pendiente ES el peso.
# Se dibuja de a 10 puntos para que se vea; de a 1 seria invisible en este rango.
x0, paso = 62, 10
y0 = intercepto_1 + peso_1 * x0
y1 = intercepto_1 + peso_1 * (x0 + paso)
ax.plot([x0, x0 + paso], [y0, y0], color=AQUA, linewidth=2.6, zorder=6)
ax.plot([x0 + paso, x0 + paso], [y0, y1], color=AQUA, linewidth=2.6, zorder=6)
ax.text(x0 + paso / 2, y0 - 3.4, f"+{paso} de reacciones", fontsize=10,
        fontweight="bold", color=AQUA, ha="center", va="top", zorder=7,
        bbox=dict(boxstyle="round,pad=0.25", facecolor=SUPERFICIE,
                  edgecolor="none", alpha=0.85))
ax.text(x0 + paso + 1.6, (y0 + y1) / 2,
        f"+{coma(peso_1 * paso, 1)}\nde OVR", fontsize=10, fontweight="bold",
        color=AQUA, ha="left", va="center", linespacing=1.3, zorder=7,
        bbox=dict(boxstyle="round,pad=0.25", facecolor=SUPERFICIE,
                  edgecolor="none", alpha=0.85))
ax.text(0.03, 0.87, f"o sea {coma(peso_1)} de OVR por cada punto: ése es el peso",
        transform=ax.transAxes, fontsize=11, fontweight="bold", color=AQUA,
        va="top", ha="left")

ax.text(0.03, 0.95, f"OVR = {coma(intercepto_1, 1)} + {coma(peso_1)} × reacciones",
        transform=ax.transAxes, fontsize=13, fontweight="bold", color=NARANJA,
        va="top", ha="left")

ax.set_xlabel("Reacciones del jugador", labelpad=8)
ax.set_ylabel("OVR del jugador")
ax.grid(True, zorder=0)
ax.set_axisbelow(True)

poner_titulo(ax, "Un peso es la pendiente de una recta",
             "Con un solo atributo se ve entero: cuánto sube el OVR por cada punto de reacciones",
             alto=(1.12, 1.04))
ax.text(0, -0.155,
        f"Cada punto es un jugador (se grafican 4.000 de los {miles(len(campo))} para que se vea la nube). "
        "La recta naranja es la que mejor pasa por el medio.\nSu pendiente es "
        f"{coma(peso_1)}: ése es el peso de reacciones cuando es el único atributo que miramos.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "21_que_es_un_peso.png")
plt.show()

## Parte 2 — Qué quiere decir "la recta que mejor pasa"

Dijimos "la recta que mejor pasa por la nube", pero eso todavía es una frase, no un
criterio. ¿Mejor según qué?

Para cualquier recta que dibujemos, cada jugador queda a cierta distancia vertical de
ella: la diferencia entre su OVR real y el que la recta le predice. **Eso es el error de
ese jugador.**

La regla es elevar cada error al cuadrado —para que los de arriba y los de abajo no se
cancelen, y para castigar más los errores grandes— y sumarlos todos. La mejor recta es
la que hace esa suma lo más chica posible. De ahí el nombre del método: **mínimos
cuadrados**.

Con 15.859 jugadores no se ven los segmentos. Así que acá tomamos 14 al azar y los
dibujamos uno por uno.

In [ ]:
pocos = campo.sample(14, random_state=7).sort_values("reactions")
predicho_pocos = intercepto_1 + peso_1 * pocos["reactions"]
errores_pocos = pocos["ovr"] - predicho_pocos

fig, ax = plt.subplots(figsize=(10, 5.8))

rango_p = np.array([pocos["reactions"].min() - 3, pocos["reactions"].max() + 3])
ax.plot(rango_p, intercepto_1 + peso_1 * rango_p, color=NARANJA, linewidth=2.4, zorder=3)

for xi, real, pred in zip(pocos["reactions"], pocos["ovr"], predicho_pocos):
    ax.plot([xi, xi], [pred, real], color=AQUA, linewidth=1.6,
            linestyle=(0, (3, 2)), zorder=4)
ax.scatter(pocos["reactions"], pocos["ovr"], s=70, color=AZUL,
           edgecolors=SUPERFICIE, linewidth=1.5, zorder=5)

suma_cuadrados = (errores_pocos ** 2).sum()
ax.text(0.03, 0.95,
        "Cada línea punteada es lo que la recta\nse equivoca con ese jugador",
        transform=ax.transAxes, fontsize=11, color=AQUA, fontweight="bold",
        va="top", ha="left", linespacing=1.4)
ax.text(0.03, 0.80,
        f"Suma de los 14 errores al cuadrado: {coma(suma_cuadrados, 1)}",
        transform=ax.transAxes, fontsize=10.5, color=TINTA_SUAVE, va="top", ha="left")

ax.set_xlabel("Reacciones del jugador", labelpad=8)
ax.set_ylabel("OVR del jugador")
ax.grid(True, zorder=0)
ax.set_axisbelow(True)

poner_titulo(ax, "«La mejor recta» quiere decir la que menos se equivoca",
             "14 jugadores tomados al azar, con su error dibujado",
             alto=(1.12, 1.04))
ax.text(0, -0.155,
        "El método se llama mínimos cuadrados: se eleva cada error al cuadrado —para que los de arriba no cancelen "
        "a los de abajo—\ny se suman todos. La recta elegida es la que hace esa suma lo más chica posible.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "22_que_es_el_error.png")
plt.show()

## Parte 3 — De dónde sale el número

Ahora sí, la pregunta de la devolución: **¿de dónde sale el 0,72?**

De ningún lado en particular. Nadie lo eligió. Es el resultado de probar todas las
pendientes posibles y quedarse con la que menos se equivoca.

Acá lo hacemos literalmente: recorremos pendientes desde 0,30 hasta 1,40, calculamos para
cada una el error cuadrático medio sobre los 15.859 jugadores, y lo graficamos. Sale una
curva con forma de U, y **el peso es el punto más bajo de esa curva**.

Es el mismo número que devuelve `LinearRegression` de scikit-learn, pero calculado a
mano para que se vea que no hay magia adentro.

In [ ]:
xv = campo["reactions"].values
yv = campo["ovr"].values

pendientes = np.linspace(0.30, 1.40, 200)
# Para cada pendiente, el mejor intercepto posible es el que centra la recta en la nube
errores_curva = np.array([
    np.mean((yv - (yv.mean() - p * xv.mean()) - p * xv) ** 2) for p in pendientes
])
mejor = pendientes[errores_curva.argmin()]

print(f"Pendiente que minimiza el error (a mano):  {mejor:.4f}")
print(f"Pendiente que devuelve scikit-learn:       {peso_1:.4f}")
print("\nError cuadrático medio para algunas pendientes:")
for p in [0.50, 0.60, 0.72, 0.90, 1.10]:
    e = np.mean((yv - (yv.mean() - p * xv.mean()) - p * xv) ** 2)
    print(f"   pendiente {coma(p)}  ->  error {coma(e, 2)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4))

# --- Izquierda: tres rectas candidatas sobre la nube ---
ax = axes[0]
ax.scatter(muestra["reactions"], muestra["ovr"], s=6, alpha=0.11,
           color=AZUL, edgecolors="none", zorder=2)

candidatas = [(0.50, GRIS, "muy plana"), (peso_1, NARANJA, "la mejor"),
              (1.10, GRIS, "muy empinada")]
# Las rectas se recortan al rango central de la nube para que no la aplasten
recorte = np.array([40, 88])
for p, color, etiqueta in candidatas:
    inter = yv.mean() - p * xv.mean()
    ax.plot(recorte, inter + p * recorte, color=color,
            linewidth=2.6 if color == NARANJA else 1.8,
            linestyle="-" if color == NARANJA else (0, (5, 3)), zorder=4)
    e = np.mean((yv - inter - p * xv) ** 2)
    ax.text(recorte[1] + 1.2, inter + p * recorte[1],
            f"{coma(p)} · error {coma(e, 1)}", fontsize=9.5,
            color=color if color == NARANJA else TINTA_TENUE,
            fontweight="bold" if color == NARANJA else "normal",
            va="center", ha="left")

ax.set_xlim(32, 112)
ax.set_ylim(40, 96)
ax.set_xlabel("Reacciones", labelpad=6)
ax.set_ylabel("OVR")
ax.grid(True, zorder=0)
ax.set_axisbelow(True)

# --- Derecha: la curva de error ---
ax = axes[1]
ax.plot(pendientes, errores_curva, color=AZUL, linewidth=2.6, zorder=3)
ax.scatter([mejor], [errores_curva.min()], s=110, color=NARANJA, zorder=5,
           edgecolors=SUPERFICIE, linewidth=2)
ax.annotate(f"el mínimo está en {coma(mejor)}",
            xy=(mejor, errores_curva.min()), xytext=(mejor, errores_curva.min() + 14),
            fontsize=11, fontweight="bold", color=NARANJA, ha="center", va="bottom",
            arrowprops=dict(arrowstyle="-|>", color=NARANJA, linewidth=1.6,
                            shrinkA=6, shrinkB=5))

for p, color, etiqueta in candidatas:
    if color == GRIS:
        e = np.mean((yv - (yv.mean() - p * xv.mean()) - p * xv) ** 2)
        ax.scatter([p], [e], s=55, color=GRIS, zorder=4,
                   edgecolors=SUPERFICIE, linewidth=1.5)

ax.set_xlabel("Pendiente que se prueba (el peso candidato)", labelpad=6)
ax.set_ylabel("Error cuadrático medio")
ax.grid(True, zorder=0)
ax.set_axisbelow(True)

poner_titulo(axes[0], "El peso no se elige: es el fondo de una curva",
             "Izquierda: tres pendientes posibles.  Derecha: el error de cada pendiente, de 0,30 a 1,40",
             alto=(1.15, 1.06))
axes[0].text(0, -0.17,
             f"Probamos 200 pendientes y medimos cuánto se equivoca cada una con los {miles(len(campo))} jugadores. "
             f"La que menos se equivoca es {coma(mejor)},\nel mismo número que devuelve scikit-learn. "
             "Eso es todo lo que hace una regresión lineal: encontrar ese mínimo.",
             transform=axes[0].transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "23_de_donde_sale_el_peso.png")
plt.show()

## Parte 4 — Dos atributos: la recta pasa a ser un plano

Hasta acá todo fue con un solo atributo. Si agregamos un segundo —pase corto— la fórmula
gana un término:

```latex
OVR = intercepto + peso_1 \times reacciones + peso_2 \times pase\ corto
```

Geométricamente ya no es una recta en un gráfico plano: es **un plano flotando en el
espacio**, y tiene dos pendientes, una en cada dirección. Cada peso es la pendiente del
plano en el eje de su atributo.

El criterio para elegirlo es exactamente el mismo que antes: el plano que deja la suma de
errores al cuadrado lo más chica posible.

In [ ]:
dos = ["reactions", "short_passing"]
modelo_2 = LinearRegression().fit(campo[dos], campo["ovr"])
pesos_2 = pd.Series(modelo_2.coef_, index=dos)
r2_2 = modelo_2.score(campo[dos], campo["ovr"])

print(f"OVR = {modelo_2.intercept_:.3f}"
      f" + {pesos_2['reactions']:.4f} × reacciones"
      f" + {pesos_2['short_passing']:.4f} × pase corto")
print(f"R² = {r2_2:.4f}   (con un solo atributo era {r2_1:.4f})")

In [ ]:
fig = plt.figure(figsize=(11, 6.0))
# El eje 3D se estira a mano: por defecto deja media figura en blanco
ax = fig.add_axes([0.02, 0.07, 0.96, 0.82], projection="3d")
ax.set_facecolor(SUPERFICIE)

rej = np.linspace(campo["reactions"].min(), campo["reactions"].max(), 14)
pej = np.linspace(campo["short_passing"].min(), campo["short_passing"].max(), 14)
RE, PE = np.meshgrid(rej, pej)
Z = modelo_2.intercept_ + pesos_2["reactions"] * RE + pesos_2["short_passing"] * PE
ax.plot_surface(RE, PE, Z, color=NARANJA, alpha=0.28, edgecolor=NARANJA,
                linewidth=0.35, antialiased=True, zorder=2)

m3d = campo.sample(700, random_state=11)
ax.scatter(m3d["reactions"], m3d["short_passing"], m3d["ovr"],
           s=13, alpha=0.5, color=AZUL, edgecolors="none", depthshade=False, zorder=4)

ax.set_xlabel(f"Reacciones  →  pendiente {coma(pesos_2['reactions'])}",
              labelpad=12, fontsize=10.5)
ax.set_ylabel(f"Pase corto  →  pendiente {coma(pesos_2['short_passing'])}",
              labelpad=12, fontsize=10.5)
ax.set_zlabel("OVR", labelpad=4, fontsize=10.5)
ax.tick_params(labelsize=9)
ax.view_init(elev=16, azim=-62)
ax.set_box_aspect((1.6, 1.6, 0.8))
for panel in (ax.xaxis, ax.yaxis, ax.zaxis):
    panel.pane.set_facecolor(SUPERFICIE)
    panel.pane.set_edgecolor(GRILLA)
ax.grid(False)

fig.text(0.05, 0.98, "Con dos atributos, la recta pasa a ser un plano",
         fontsize=14, fontweight="bold", color=TINTA, va="top", ha="left")
fig.text(0.05, 0.935,
         "Dos atributos, dos pendientes: una en cada dirección del plano",
         fontsize=10.5, color=TINTA_SUAVE, va="top", ha="left")
fig.text(0.05, 0.085,
         f"OVR = {coma(modelo_2.intercept_, 1)} + {coma(pesos_2['reactions'])} × reacciones"
         f" + {coma(pesos_2['short_passing'])} × pase corto",
         fontsize=12.5, fontweight="bold", color=NARANJA, va="top", ha="left")
fig.text(0.05, 0.035,
         "Cada peso sigue siendo una pendiente y se sigue eligiendo igual: el plano que menos se equivoca. "
         f"Con dos atributos el R² sube de {coma(r2_1, 3)} a {coma(r2_2, 3)}.",
         fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "24_dos_atributos_un_plano.png")
plt.show()

## Parte 5 — De dos a veintinueve: por qué el peso baja

Con 29 atributos ya no hay dibujo posible: sería un objeto de 29 dimensiones. Pero la
idea no cambia en nada. Son 29 pendientes, elegidas con el mismo criterio.

Ahora bien, acá aparece algo que conviene explicar, porque es la parte que más confunde:
**el peso de un atributo cambia según cuántos atributos haya en el modelo.**

Reacciones pesaba 0,72 cuando era el único. Al agregar pase corto bajó a 0,51. Con los 29
queda en 0,29. Y en el modelo entrenado sólo con delanteros, 0,09.

No es que el número esté mal en un caso y bien en otro: **miden cosas distintas.** Cuando
reacciones es el único atributo, su pendiente se lleva puesto el crédito de todo lo que
viene con ella —un jugador con buenas reacciones suele tener también buen control y buen
pase—. Cuando los otros 28 están en el modelo, cada uno reclama su parte, y a reacciones
le queda sólo lo que aporta ella sola, con todo lo demás fijo.

**Ése es el número que está en la tabla de la etapa 3.** El gráfico siguiente muestra la
cadena completa, de 0,72 hasta 0,09.

In [ ]:
cadena = [
    (["reactions"], "Sólo reacciones", "1 atributo"),
    (["reactions", "short_passing"], "+ pase corto", "2 atributos"),
    (["reactions", "short_passing", "ball_control", "finishing", "positioning"],
     "+ control, definición\ny posicionamiento", "5 atributos"),
    (ATRIBUTOS, "Los 29 in-game stats", "29 atributos"),
]

filas = []
for columnas, etiqueta, cuantos in cadena:
    m = LinearRegression().fit(campo[columnas], campo["ovr"])
    filas.append({
        "etiqueta": etiqueta, "cuantos": cuantos,
        "peso": pd.Series(m.coef_, index=columnas)["reactions"],
        "r2": m.score(campo[columnas], campo["ovr"]),
    })

delanteros = campo[campo["position"] == "ST"]
m_st = LinearRegression().fit(delanteros[ATRIBUTOS], delanteros["ovr"])
filas.append({
    "etiqueta": "Los 29, sólo\nentre delanteros", "cuantos": "29 atributos · sólo ST",
    "peso": pd.Series(m_st.coef_, index=ATRIBUTOS)["reactions"],
    "r2": m_st.score(delanteros[ATRIBUTOS], delanteros["ovr"]),
})

tabla_cadena = pd.DataFrame(filas)
print(tabla_cadena[["cuantos", "peso", "r2"]].round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.8))

posiciones = range(len(tabla_cadena))
colores = [AZUL] * (len(tabla_cadena) - 1) + [AQUA]
ax.bar(posiciones, tabla_cadena["peso"], color=colores, width=0.6, zorder=3)

for i, fila in tabla_cadena.iterrows():
    ax.text(i, fila["peso"] + 0.018, coma(fila["peso"]), ha="center", va="bottom",
            fontsize=13, fontweight="bold",
            color=AQUA if i == len(tabla_cadena) - 1 else AZUL)
    ax.text(i, -0.055, fila["cuantos"], ha="center", va="top",
            fontsize=9, color=TINTA_TENUE)
    ax.text(i, fila["peso"] / 2, f"R² {coma(fila['r2'], 3)}", ha="center", va="center",
            fontsize=9.5, color="white", fontweight="bold")

ax.set_xticks(list(posiciones), tabla_cadena["etiqueta"], fontsize=10,
              color=TINTA_SUAVE, linespacing=1.3)
ax.tick_params(axis="x", pad=26)
ax.set_ylim(0, 0.85)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8], ["0", "0,2", "0,4", "0,6", "0,8"])
ax.set_ylabel("Peso de reacciones sobre el OVR")
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

ax.annotate("", xy=(4.3, 0.12), xytext=(-0.3, 0.78),
            arrowprops=dict(arrowstyle="-|>", color=NARANJA, linewidth=2,
                            connectionstyle="arc3,rad=-0.18"), zorder=6)
ax.text(2.1, 0.60, "cuantos más atributos,\nmenos se lleva cada uno",
        fontsize=11, fontweight="bold", color=NARANJA, ha="center", va="center",
        linespacing=1.4)

poner_titulo(ax, "El mismo atributo, cinco pesos distintos",
             "Reacciones pesa 0,72 cuando está sola y 0,09 en el modelo de los delanteros",
             alto=(1.13, 1.045))
ax.text(0, -0.24,
        "Ninguno de los cinco está mal: miden cosas distintas. Con un solo atributo, la pendiente se lleva el crédito de "
        "todo lo que\nviene junto con las reacciones. Con los 29 adentro, a cada uno le queda sólo lo que aporta por su "
        "cuenta — y ése es\nel número que aparece en la tabla de la etapa 3.",
        transform=ax.transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "25_el_peso_se_reparte.png")
plt.show()

## Parte 6 — Por qué un modelo por cada posición

Falta una última pieza. En la etapa 3 entrenamos un modelo separado para cada posición, y
la tabla de pesos tiene una columna por puesto. ¿Por qué?

Porque la pendiente del mismo atributo es distinta según la posición. Y eso se puede ver
en un gráfico, sin fórmulas: abajo está la definición contra el OVR, primero entre
delanteros y después entre centrales. Es el mismo atributo y los mismos ejes; lo único
que cambia es qué jugadores se miran.

Entre delanteros la recta sube marcadamente. Entre centrales es casi horizontal: saber
cuánto define un central no dice casi nada sobre su OVR.

**Una sola recta para los dos grupos no puede representar a ninguno.** Por eso hay una
regresión por posición, y por eso la tabla tiene una columna por puesto.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4), sharey=True)

for ax, (pos, nombre_pos) in zip(axes, [("ST", "Delanteros (ST)"), ("CB", "Centrales (CB)")]):
    sub = campo[campo["position"] == pos]
    ms = LinearRegression().fit(sub[["finishing"]], sub["ovr"])
    mm = LinearRegression().fit(sub[ATRIBUTOS], sub["ovr"])
    peso_multiple = pd.Series(mm.coef_, index=ATRIBUTOS)["finishing"]

    ax.scatter(sub["finishing"], sub["ovr"], s=9, alpha=0.22,
               color=AZUL, edgecolors="none", zorder=2)
    rf = np.array([sub["finishing"].min(), sub["finishing"].max()])
    ax.plot(rf, ms.intercept_ + ms.coef_[0] * rf, color=NARANJA, linewidth=2.8, zorder=4)

    # Fondo claro detras del bloque de texto: cae sobre la nube de puntos
    caja = dict(boxstyle="round,pad=0.5", facecolor=SUPERFICIE,
                edgecolor=GRILLA, linewidth=1)
    ax.text(0.04, 0.96,
            f"{nombre_pos}\n{miles(len(sub))} jugadores\n"
            f"pendiente {coma(ms.coef_[0])}\n"
            f"peso con los 29: {coma(peso_multiple)}",
            transform=ax.transAxes, fontsize=10.5, color=TINTA_SUAVE,
            va="top", ha="left", linespacing=1.6, bbox=caja, zorder=8)

    ax.set_xlabel("Definición del jugador", labelpad=8)
    ax.grid(True, zorder=0)
    ax.set_axisbelow(True)

axes[0].set_ylabel("OVR del jugador")

poner_titulo(axes[0], "El mismo atributo pesa distinto según la posición",
             "Definición contra OVR: marcada entre delanteros, casi horizontal entre centrales",
             alto=(1.15, 1.06))
axes[0].text(0, -0.17,
             "Es el mismo atributo y los mismos ejes: lo único que cambia es qué jugadores se miran. Una sola recta para "
             "los dos grupos\nno podría representar a ninguno, y por eso hay una regresión por posición. "
             "La tabla de la etapa 3 es exactamente eso:\nuna columna de pendientes por cada puesto.",
             transform=axes[0].transAxes, fontsize=9.5, color=TINTA_TENUE, va="top", ha="left")

guardar(fig, "26_misma_variable_distinta_pendiente.png")
plt.show()

## Cierre — la tabla de la etapa 3, leída de nuevo

Con el recorrido hecho, cada celda de aquella tabla se puede nombrar sin ambigüedad:

> La celda **Definición / ST** vale **0,19**. Es la pendiente de la definición en la
> regresión entrenada únicamente con los 2.476 delanteros, con los otros 28 atributos
> dentro del modelo. Significa que, entre delanteros y dejando todo lo demás fijo, cada
> punto de definición agrega 0,19 puntos de OVR.

Y las tres propiedades que habíamos afirmado en la etapa 3 se siguen de ahí:

- **Los pesos suman 1,00** en cada posición. Eso es lo que permite afirmar que el OVR es
  un promedio ponderado, y no cualquier otra combinación.
- **Reacciones es el único atributo con peso en las nueve posiciones** (entre 0,05 y
  0,09). Por eso en la etapa 2 correlacionaba 0,90 en todas.
- **Un peso de 0,00 no quiere decir que el atributo sea inútil**, sino que no aporta nada
  *una vez que los otros 28 ya están en el modelo*. La definición de un central mide algo
  real; lo que dice el modelo es que no interviene en su OVR.

In [ ]:
POSICIONES = ["ST", "RW", "CAM", "CM", "CDM", "LM", "RB", "LB", "CB"]

resumen = []
for pos in POSICIONES:
    sub = campo[campo["position"] == pos]
    m = LinearRegression().fit(sub[ATRIBUTOS], sub["ovr"])
    p = pd.Series(m.coef_, index=ATRIBUTOS)
    resumen.append({
        "posición": pos,
        "jugadores": len(sub),
        "suma de pesos": round(p.sum(), 3),
        "intercepto": round(m.intercept_, 2),
        "peso de reacciones": round(p["reactions"], 3),
        "atributo de mayor peso": NOMBRES.get(p.idxmax(), p.idxmax()),
        "su peso": round(p.max(), 3),
    })

tabla_resumen = pd.DataFrame(resumen)
print(tabla_resumen.to_string(index=False))
print("\nLos pesos suman 1,00 en las nueve posiciones y reacciones nunca vale cero.")